# Perform radiometric calibration
Input Level 1.5 FUV spectrograph data, outputs Level 1.6 data with physical units

Also try getting DN2PHOT and AREA_SG from `irispy` (see Google Colab notebook)

The next step after this is assembling the mosaic using `build_and_save_mosaic.ipynb`.

In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline
# %matplotlib notebook

import pathlib as pl
import numpy as np
import pickle
from mpl_toolkits.axes_grid1 import ImageGrid
import matplotlib.pyplot as plt
import astropy.units as u
import astropy.constants as const
from astropy.io import fits
from astropy.nddata import block_reduce
import astropy.wcs
import scipy.interpolate
import warnings

from iris_mosaics import read_sg_image, read_sg_image_old
import iris_mosaics as iris_fdm
from IPython.display import display, Math, Markdown
from astropy.visualization import quantity_support
quantity_support()

In [ ]:
# date_string = '20140324'
# date_string = '20190912'
date_string = '20240811'
# path = pl.Path(fr'D:\IRIS data\deep_mosaics\{date_string}\level_15')
path = pl.Path(fr'D:\IRIS data\deep_mosaics\{date_string}\level_15_rebinned')
files = list(path.glob('*.fits'))
# files = files[:10]

file_num = 0
sg_wcs, hdu, _ = read_sg_image(files[file_num],'fuv2')
sg_img = hdu[0].data

In [ ]:
sg_wavelength_full = np.squeeze(sg_wcs.array_index_to_world(*np.indices((1, 1, sg_img.shape[1])))[0].to(u.Angstrom))

#### Rebin the Aug 11, 2024 mosaic such that the spectral direction has the same pixels as normal deep mosaics
Skip to DN2PHOT_SG for normal deep mosaics

In [ ]:
def nansum_keepnan(a, axis):
    s = np.nansum(a, axis=axis)
    s[np.all(np.isnan(a), axis=axis)] = np.nan   # all-NaN block -> stay NaN
    return s

def bin_wcs_pixel_axis(wcs, pix_axis_fits, factor):
    """Copy of `wcs` with pixel axis `pix_axis_fits` (1-indexed) binned by
    `factor`. CDELT for the matching world axis reads the binned step;
    the full transform is exact, cross-terms included."""
    w = wcs.deepcopy()
    j = pix_axis_fits - 1

    crpix = w.wcs.crpix.copy()
    crpix[j] = (crpix[j] - 0.5) / factor + 0.5
    w.wcs.crpix = crpix

    if w.wcs.has_cd():
        cd = w.wcs.cd.copy(); cd[:, j] *= factor; w.wcs.cd = cd
    else:
        pc = w.wcs.get_pc().copy()
        pc[:, j] *= factor
        pc[j, :] /= factor
        w.wcs.pc = pc
        cdelt = w.wcs.cdelt.copy()
        cdelt[j] *= factor
        w.wcs.cdelt = cdelt

    ps = w.pixel_shape
    if ps is not None and all(s is not None for s in ps):
        ps = list(ps) + [1] * (w.naxis - len(ps))   # pad degenerate solar-x axis
        ps[j] //= factor                            # 2072 -> 1036
        w.pixel_shape = tuple(ps)
    return w

In [ ]:
outdir = files[0].parent.parent / 'level_15_rebinned'
outdir.mkdir(parents=True, exist_ok=True)

sg_img_set = np.empty((len(files), 548, 1036), dtype=np.float32)
sg_wcs = []

for i, file in enumerate(files):
    w_fuv2, hdu, _ = read_sg_image(file, 'fuv2')        # w_fuv2.wcs.alt == 'A'
    hdr = hdu[0].header
    w_fuv1 = astropy.wcs.WCS(hdr)                       # primary description

    sg_img_set[i] = block_reduce(hdu[0].data, (1, 2), func=nansum_keepnan)

    w1_new = bin_wcs_pixel_axis(w_fuv1, pix_axis_fits=1, factor=2)
    w2_new = bin_wcs_pixel_axis(w_fuv2, pix_axis_fits=1, factor=2)
    sg_wcs.append(w2_new)

    hdu[0].data = sg_img_set[i]
    hdr.update(w1_new.to_header())                      # -> CRPIX1, CDELT1, ...
    hdr.update(w2_new.to_header())                      # -> CRPIX1A, CDELT1A, ... automatically
    hdr['HISTORY'] = 'Spectral axis (NAXIS1) summed x2, nansum, all-NaN blocks stay NaN'
    hdr['HISTORY'] = 'Both WCS descriptions (primary/FUV1, A/FUV2) rebinned'
    hdu.writeto(outdir / file.name, overwrite=True)
    hdu.close()
    warnings.filterwarnings('ignore', category=astropy.wcs.FITSFixedWarning)

Read in single rebinned image

In [ ]:
path_rebinned = pl.Path(fr'D:\IRIS data\deep_mosaics\{date_string}\level_15_rebinned')
files_rebinned = list(path_rebinned.glob('*.fits'))

file_num = ~0
sg_wcs, hdu, _ = read_sg_image(files_rebinned[file_num],'fuv2')
sg_img = hdu[0].data

In [ ]:
sg_wavelength_full = np.squeeze(sg_wcs.array_index_to_world(*np.indices((1, 1, sg_img.shape[1])))[0].to(u.Angstrom))

Need the effective area, which is retrieved by `get_dn2phot_and_area_sg.pro`. This prints the FUV DN2PHOT_SG value and saves an array with the wavelength (in nm) and corresponding effective area for that wavelength.
Doesn't seem to change too much over the whole mosaic (max difference between first and last image was 0.00012), so may only need to use effective area from one SG image.

DN2PHOT_SG

In [ ]:
dn2phot_sg = 4

Effective area ($A_\text{eff}$) in cm$^2$

In [ ]:
# These fits files were obtained from your google colab notebook that uses irispy-lmsal to run `get_response`
file_area = pl.Path(fr'D:\IRIS data\deep_mosaics\{date_string}\{date_string}_area_sg_fuv.fits')
file_lambda = pl.Path(fr'D:\IRIS data\deep_mosaics\{date_string}\area_sg_fuv_lambda.fits')

area_sg_full = fits.open(file_area)[0].data * u.cm**2
area_sg_full_wl = (fits.open(file_lambda)[0].data * u.nm).to(u.Angstrom)

In [ ]:
# path_area = pl.Path(r'D:\IRIS data\deep_mosaics\20190912\area_sg')
# # path_area = pl.Path(iris_fdm.__file__).parent / 'data' / 'unstitched_mosaic' / 'science_oct_20_2019' / 'area_sg'
# files_area = list(path_area.glob('*.fits'))
# files_area = files_area[0]
#
# _, hdu_area, _ = read_sg_image(files_area,'fuv2')
# area_sg_img = hdu_area[0].data
#
# area_sg_full_wl = (area_sg_img[0,:] * u.nm).to(u.Angstrom)
# area_sg_full = area_sg_img[1,:] * u.cm**2

In [ ]:
plt.figure(figsize=(7,4))
plt.plot(area_sg_full_wl, area_sg_full)
plt.xlim(1320, 1420) * u.Angstrom;

Interpolate the effective area to our wavelength values and extrapolate where the effective area is zero (the Si IV lines aren't in the zero region anyway).

In [ ]:
area_sg_mask = area_sg_full!=0
area_sg_interp = scipy.interpolate.interp1d(area_sg_full_wl[area_sg_mask], area_sg_full[area_sg_mask])
area_sg = area_sg_interp(sg_wavelength_full) * u.cm**2

In [ ]:
plt.figure(figsize=(7,4))
plt.plot(area_sg_full_wl, area_sg_full)
plt.plot(sg_wavelength_full, area_sg)
plt.xlim(sg_wavelength_full.min(), sg_wavelength_full.max());

Energy

In [ ]:
energy = const.h * const.c / sg_wavelength_full
energy = energy.to(u.erg)
energy[0]

Spatial pixel in radians

In [ ]:
pix_spatial = (sg_wcs.wcs.cdelt[1] * u.deg).to(u.rad)
pix_spatial

Spectral pixel in Angstroms

In [ ]:
pix_spectral = (sg_wcs.wcs.cdelt[0] * u.m).to(u.Angstrom)
pix_spectral

Slit width in radians

In [ ]:
w_slit = (sg_wcs.wcs.cdelt[2] * u.deg).to(u.rad)
w_slit

Exposure time in seconds

In [ ]:
exptime = hdu[0].header['EXPTIME'] * u.s
exptime

Conversion factor to go from DN to flux. Should be in erg s$^{-1}$ cm$^{-2}$ Å$^{-1}$ sr$^{-1}$

In [ ]:
dn2flux_conversion = (energy * dn2phot_sg) / (area_sg * pix_spectral * exptime * (pix_spatial * w_slit).to(u.sr))
plt.figure()
plt.plot(sg_wavelength_full, dn2flux_conversion);

Convert to W m$^{-2}$ nm$^{-1}$ sr$^{-1}$

In [ ]:
dn2flux_conversion = dn2flux_conversion.to(u.W / (u.m**2 * u.nm * u.sr))
plt.figure()
plt.plot(sg_wavelength_full, dn2flux_conversion);

Apply conversion factor

In [ ]:
sg_img = sg_img * dn2flux_conversion

Pick a point to test

In [ ]:
sg_img[200, 200]

Define slice to just check out the Si IV lines

In [ ]:
# si_iv_sl = ..., slice(5,~12), slice(732,~13)
si_iv_sl = ..., slice(5,~12), slice(732,~13)
# si_iv_sl = slice(None), slice(720,None)

num_y = sg_img[si_iv_sl].shape[0]
num_x = sg_img[si_iv_sl].shape[1]

plt.figure(figsize=(7,4))
plt.imshow(sg_img[si_iv_sl].value,
           vmin=-10,
           vmax=100)
plt.colorbar()

Rather than saving a whole other mosaic, just save this wavelength-dependent conversion factor and apply when needed.

In [ ]:
# Save values and units separately in fits file
# Create the HDU object and assign the unit to 'BUNIT'
unit_string = dn2flux_conversion.unit.to_string('fits')  # Ensures FITS compliance
hdu = fits.PrimaryHDU(data=dn2flux_conversion.value)
hdu.header['BUNIT'] = (unit_string, 'Physical units of the data')

# Wavelength as a second extension
unit_string = sg_wavelength_full.unit.to_string('fits')
wave_hdu = fits.ImageHDU(data=sg_wavelength_full.value, name='WAVELENGTH')
wave_hdu.header['BUNIT'] = (unit_string, 'Wavelength units')

# Path to save to
path_conversionfactor = pl.Path(fr'D:\IRIS data\deep_mosaics\{date_string}\{date_string}_radiometric_calibration_conversion_factor.fits')

# Write to disk
fits.HDUList([hdu, wave_hdu]).writeto(path_conversionfactor, overwrite=True)

Apply this conversion from DN to W m^-2^ nm^-1^ sr^-1^ to each spectrograph image and save

In [ ]:
# num_imgs = len(files)
# si_iv_sg_calibrated = np.empty((num_imgs, num_y, num_x))
#
# for file_index, file in enumerate(files):
#
#     w_i, hdu_i = read_sg_image(file,'fuv2')
#     img = hdu_i[0].data
#
#     img = img * (energy * dn2phot_sg) / (area_sg * pix_spectral * exptime * (pix_spatial * w_slit).to(u.sr))
#     img = img.to(u.W / (u.m**2 * u.nm * u.sr))
#
#     #************** UNCOMMENT FOR NORMAL OPERATIONS **************
#     # Save data
#     # hdu_i[0].data = img
#     # new_file = file.parent.parent / 'level_16' / file.name
#     # hdu_i.writeto(new_file, overwrite=True)
#
#     si_iv_sg_calibrated[file_index] = img[si_iv_sl]

Next, account for the step size and any other data we crop out in order to get somewhat accurate total spectral flux from the Sun

In [ ]:
# IRIS raster step size
step_size = 2 * u.arcsec
factor_1 = (step_size / w_slit.to(u.arcsec)).value

# # Cut out data that was in the FUV shadow, set y-pixel value
# data_cut_off = 400
# mask = slice(None, data_cut_off), slice(None)
# factor_2 = np.isfinite(si_iv_sg_calibrated[0]).sum() / np.isfinite(si_iv_sg_calibrated[0][mask]).sum()

# Apply correction factors
# si_iv_spectrum_calibrated = np.nansum(si_iv_sg_calibrated[:,:data_cut_off,:], axis=(0,1)) * factor_1 * factor_2
si_iv_spectrum_calibrated = np.nansum(si_iv_sg_calibrated, axis=(0,1)) * factor_1

# Attach units (removed in last step..?)
si_iv_spectrum_calibrated = si_iv_spectrum_calibrated * u.W / (u.m**2 * u.nm * u.sr)

In [ ]:
plt.figure(figsize=(14,7))
plt.plot(sg_wavelength_full[si_iv_sl[-1]], si_iv_spectrum_calibrated)
plt.axhline(y=0, color='r', linewidth=1, linestyle='dotted')
plt.ylim((-si_iv_spectrum_calibrated.max()/20,si_iv_spectrum_calibrated.max()/2))

# Plot line labels
si_iv_1394 = 1393.755 * u.Angstrom
si_iv_1403 = 1402.770 * u.Angstrom
plt.axvline(x=si_iv_1394, color='gray', linewidth='0.5')
plt.axvline(x=si_iv_1403, color='gray', linewidth='0.5')
plt.text(x=si_iv_1394, y=8.25e8, s='Si IV', rotation='vertical', ha='right', size=9)
plt.text(x=si_iv_1403, y=3.8e8, s='Si IV', rotation='vertical', ha='right', size=9)

line_list_1394 = [1392.149, 1392.588, 1392.817, 1393.330] * u.angstrom
label_list_1394 = ['Fe II', 'S I', 'Fe II', 'Ni II']
for line, label in zip(line_list_1394, label_list_1394):
    plt.axvline(x=line, color='gray', linewidth='0.5')
    plt.text(x=line, y=1e8, s=label, rotation='vertical', ha='right', size=9)

line_list_1403 = [1398.758, 1399.026, 1399.774, 1401.156, 1401.514, 1404.779, 1405.6081, 1406.043] * u.angstrom
label_list_1403 = ['Ni II', 'Ni II', 'O IV', 'O IV', 'S I', 'O IV', 'Fe II', 'S IV']
for line, label in zip(line_list_1403, label_list_1403):
    plt.axvline(x=line, color='gray', linewidth='0.5')
    plt.text(x=line, y=1e8, s=label, rotation='vertical', ha='right', size=9)

plt.axvline(x=1393.08 * u.angstrom, color='red', linewidth='0.5')

# plt.savefig('disk_spectrum_level_16_lmsal.png', dpi=300, transparent=True)